# M0 · simulation — hand playground

Get comfortable with the **BrainCo Revo 2** joint space in Drake + Meshcat, where nothing can
break. The hardware twin of this notebook is [`../physical/hand_playground.ipynb`](../physical/hand_playground.ipynb).

**Stack:** `pydrake` (MultibodyPlant, kinematics only) → `Meshcat` (3D view) → `ipywidgets`
(sliders). Backend code: [`sim_hand.py`](sim_hand.py). No hardware, no serial port, no SDK.

**Shared with the physical stack:** only [`../hand_model.py`](../hand_model.py) — the finger
table and the normalized 6-vector pose. Poses you save here land in `../poses.json`, which the
physical notebooks replay unchanged. That file is the whole interface between the two stacks.

| # | Name | URDF drive joint | Range | Coupled joint |
|---|------|------------------|-------|---------------|
| 0 | `thumb`     | `right_thumb_proximal_joint`   | 0–59° | `right_thumb_distal_joint` ×1.0 |
| 1 | `thumb_aux` | `right_thumb_metacarpal_joint` | 0–90° | — (abduction: swings the thumb across the palm) |
| 2 | `index`     | `right_index_proximal_joint`   | 0–81° | `right_index_distal_joint` ×1.155 |
| 3 | `middle`    | `right_middle_proximal_joint`  | 0–81° | `right_middle_distal_joint` ×1.155 |
| 4 | `ring`      | `right_ring_proximal_joint`    | 0–81° | `right_ring_distal_joint` ×1.155 |
| 5 | `pinky`     | `right_pinky_proximal_joint`   | 0–81° | `right_pinky_distal_joint` ×1.155 |

0 = open, 1 = closed. The Revo 2 has 6 actuated joints and 11 DOF: each distal joint is
mechanically coupled to its proximal joint. Drake ignores the URDF's `<mimic>` tags on this
plant (they need a discrete SAP plant), so `pose_to_joint_angles` applies the coupling itself.

In [ ]:
import sys
from pathlib import Path

# Locate src/m0 whether the kernel started in this folder, in the repo root, or anywhere between.
# Matched by content, not by folder name, so it survives m0/M0 casing differences between
# a case-insensitive mac and the case-sensitive robot PC.
_here = [Path.cwd(), *Path.cwd().parents]
_candidates = [*_here, *(p / "src" / name for p in _here for name in ("m0", "M0"))]
M0_DIR = next((p for p in _candidates if (p / "hand_model.py").exists()), None)
if M0_DIR is None:
    raise RuntimeError(f"could not find src/m0 from {Path.cwd()}; open this notebook from its own folder")
for _path in (M0_DIR, M0_DIR / "simulation"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

import numpy as np

from hand_model import FINGERS, as_pose, describe, load_poses
from sim_hand import DEFAULT_BRICK, SimHand, SimHandUI, launch_meshcat

print(describe(0.0))
print(describe({"thumb": 0.5, "index": 0.8}))

## Launch

The simulation **and** its UI are built in one cell on purpose: re-running only half of it is
the classic way to end up with sliders that move nothing (see Troubleshooting at the bottom).
Open the Meshcat link that appears in the panel.

In [ ]:
meshcat = launch_meshcat()
sim = SimHand(meshcat, brick_urdf=DEFAULT_BRICK)
ui = SimHandUI(sim)
ui

## Drive it from code

The UI and the code path are the same thing — `ui.set_pose` moves the sliders, which moves the
hand, so the panel never lies about what the model is doing.

In [ ]:
ui.set_pose({"thumb": 0.5, "thumb_aux": 0.8, "index": 0.5})

print(describe(sim.pose))
print(f"thumb-index gap: {sim.gap_mm():.1f} mm")
for joint, angle in sim.joint_angles().items():
    print(f"  {joint:32s} {angle:6.1f} deg")

### Scripted sequences

Interpolating between poses is all the sim needs (`time.sleep` between publishes). The physical
stack has the same idea in `RealHand.move_to`, but there it also has to keep re-sending the
target at 50 Hz — a difference worth internalising before you touch the hardware.

In [ ]:
import time

POSES = load_poses()


def play(sequence, duration=1.0, hold=0.4, steps=30):
    for name in sequence:
        print(f"-> {name}")
        start, end = sim.pose.copy(), as_pose(POSES[name])
        for alpha in np.linspace(0.0, 1.0, steps):
            sim.set_pose(start + alpha * (end - start))   # publish only; cheap
            time.sleep(duration / steps)
        ui.set_pose(end)                                  # then resync the sliders
        time.sleep(hold)


play(["open", "pinch", "open", "lego_pinch", "open"])

### Where the fingertips actually are

Positions are in the hand's base frame (metres): `+z` along the fingers, `+x` out of the palm,
`+y` toward the thumb. M1 needs these to reason about where a brick ends up relative to the TCP
defined in [`../../common/scene.py`](../../common/scene.py).

In [ ]:
for finger in FINGERS:
    if finger.name == "thumb_aux":
        continue          # abduction of the same thumb - it shares the thumb fingertip
    p = sim.fingertip_position(finger.name)
    print(f"{finger.name:10s} x={p[0]: .4f}  y={p[1]: .4f}  z={p[2]: .4f}")

print()
print(f"thumb-index  {sim.gap_mm('thumb', 'index'):6.1f} mm")
print(f"thumb-middle {sim.gap_mm('thumb', 'middle'):6.1f} mm")
print(f"index-middle {sim.gap_mm('index', 'middle'):6.1f} mm")

## Next

- [`grasp_poses.ipynb`](grasp_poses.ipynb) — size a pinch against real brick dimensions and
  save the poses M1 will replay.
- [`../physical/hand_playground.ipynb`](../physical/hand_playground.ipynb) — the same sliders,
  on the real hand.

## Troubleshooting

**The sliders move but the hand in Meshcat does not.** In order of likelihood:

1. **You are looking at an old Meshcat tab.** A second `Meshcat()` grabs the next free port,
   so an old tab on `:7000` will happily show a frozen hand while `:7001` is the live one.
   Use the link inside the UI panel — it always points at the server this kernel owns.
2. **The UI is driving a sim you replaced.** Re-running the `SimHand(...)` cell builds a new
   model and clears Meshcat; a UI built earlier still holds the old one and publishes into a
   scene that no longer exists. That is why the launch cell builds the sim *and* the UI
   together — re-run **that** cell, not one half of it.
3. **The scene is empty / the tab connected too late.** Hit **Refresh view**, which re-sends
   the whole scene graph to every connected browser.
4. **A callback is throwing.** ipywidgets swallows exceptions from handlers, so `SimHandUI`
   catches them and prints the traceback in the red panel at the bottom. Also check the
   `slider updates:` counter — if it does not increase while you drag, the widget events are
   not reaching Python at all (JupyterLab needs `ipywidgets` installed in the *same*
   environment as the kernel; `conda activate int2026` before `jupyter lab`).